# Preprocessing

## Package Imports

In [1]:
import numpy as np
import pandas as pd
from pickle import dump, load
import os

## Dataset Loading

In [2]:
with open('data/eda/data.pkl', 'rb') as file:
    df = load(file)

display(df.head())

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
8217,6233,15718242,Wollstonecraft,725,Germany,Female,47,1,104887.43,86622.56,1
1664,548,15720187,Han,479,Germany,Female,30,7,143964.36,41879.99,0
3599,8343,15773876,Tung,655,France,Female,34,3,0.00,159638.77,0
5540,9437,15771000,Powell,684,France,Male,38,4,0.00,75609.84,0
0,747,15787619,Hsieh,844,France,Male,18,2,160980.03,145936.28,0


## Defense Against Missing Predictor

In this context, we are going to exclude RowNumber, CustomerId, and Surname. However, input column might have more columns than our input here.

In [3]:
mp_check = ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'EstimatedSalary', 'Exited']
cols = list(df.columns)
not_exist = []
for i in mp_check:
    if i not in cols:
        not_exist.append(i)

if len(not_exist)>0:
    raise ValueError(f"Some columns are missing: {not_exist}")
else:    
    print("All required columns exist.")

All required columns exist.


## Handling Against Wrong Data Type

Here we also include RowNumber, CustomerId, and Surname

In [4]:
wt_check = {
    'RowNumber': 'int64',
    'CustomerId': 'int64',
    'Surname': 'object',
    'CreditScore': 'int64',
    'Geography': 'object',
    'Gender': 'object',
    'Age': 'int64',
    'Tenure': 'int64',
    'Balance': 'float64',
    'EstimatedSalary': 'float64',
    'Exited': 'int64'
}

for i in cols:
    if df[i].dtype != wt_check[i]:
        print(f'Column {i} has wrong data type')

Sometimes the issue comes from several entries that not match intended data types. Heres some approaches to deal with them.

* Numericals on categorical features would be set as NaN. We will deal with NaN categoricals on modelling phase.

In [5]:
def str_check(x):
    if isinstance(x, str):
        return x
    else:
        return np.nan
    
df['Surname'] = df['Surname'].apply(str_check)
df['Geography'] = df['Geography'].apply(str_check)
df['Gender'] = df['Gender'].apply(str_check)

* Strings on numerical features is handled by `pd.to_numeric`. Note that this would work with '56' but not something like 'Age: 56' or 'FiftySix'

In [6]:
df['RowNumber'] = pd.to_numeric(df['RowNumber'], errors='coerce').astype('int64')
df['CustomerId'] = pd.to_numeric(df['CustomerId'], errors='coerce').astype('int64')
df['CreditScore'] = pd.to_numeric(df['CreditScore'], errors='coerce').astype('int64')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce').astype('int64')
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce').astype('int64')
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')
df['EstimatedSalary'] = pd.to_numeric(df['EstimatedSalary'], errors='coerce')
df['Exited'] = pd.to_numeric(df['Exited'], errors='coerce').astype('int64')

In [7]:
## check if our data type match our requirement
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 7500 entries, 8217 to 5431
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        7500 non-null   int64  
 1   CustomerId       7500 non-null   int64  
 2   Surname          7500 non-null   object 
 3   CreditScore      7500 non-null   int64  
 4   Geography        7500 non-null   object 
 5   Gender           7500 non-null   object 
 6   Age              7500 non-null   int64  
 7   Tenure           7500 non-null   int64  
 8   Balance          7500 non-null   float64
 9   EstimatedSalary  7500 non-null   float64
 10  Exited           7500 non-null   int64  
dtypes: float64(2), int64(6), object(3)
memory usage: 703.1+ KB
None


## Handling against missing data and outliers

For missing data:
* A data is considered to be missing if it is missing outside of RowNumber
* A row with missing data on CustomerId, Surname, and/or Exited would be removed
* Missing data on CreditScore, Age, Tenure, Balance, and EstimatedSalary would be ignored since `HistGradientBoostingClassifier` has native NaN support
* Entries not recognized on Geograpahy and Gender would be set as NaN, despite being well-written. This part would be explained in more detail during modelling phase

Note that we have no missing data at EDA phase, so no special treatment against that here.

For outlier data, here we simply remove it from our loaded data. Note that raw data can be accessed on data/eda

In [8]:
## load outlier data
with open('data/eda/out_data.pkl', 'rb') as file:
    out_df = load(file)

display(out_df.head())

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
9797,7140,15805212,Black,806,France,Female,67,1,0.0,103945.58,0
9873,3889,15576094,Sung,743,France,Male,71,0,0.0,29837.65,0
9841,7625,15587443,Akudinobi,728,France,Female,69,1,0.0,131804.86,0
9976,4591,15680167,Thomson,635,France,Female,78,6,47536.4,119400.08,0
9668,7728,15612729,Chidiebere,681,France,Female,63,7,0.0,55054.48,0


In [9]:
df_diff = df.merge(out_df, how='left',
                   left_index=True, right_index=True,
                   indicator=True)

in_col = [i+'_y' for i in cols]
in_col.append('_merge')

in_df = df_diff[df_diff['_merge'] == 'left_only']
in_df = in_df.drop(in_col, axis=1)

in_col = [i+'_x' for i in cols]
in_dict = dict(zip(in_col, cols))

in_df = in_df.rename(in_dict, axis=1)
display(in_df)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary,Exited
8217,6233,15718242,Wollstonecraft,725,Germany,Female,47,1,104887.43,86622.56,1
1664,548,15720187,Han,479,Germany,Female,30,7,143964.36,41879.99,0
3599,8343,15773876,Tung,655,France,Female,34,3,0.00,159638.77,0
5540,9437,15771000,Powell,684,France,Male,38,4,0.00,75609.84,0
0,747,15787619,Hsieh,844,France,Male,18,2,160980.03,145936.28,0
...,...,...,...,...,...,...,...,...,...,...,...
6437,414,15801559,Chiang,693,Germany,Female,41,9,181461.48,187929.43,1
4000,6739,15612358,Christie,573,Germany,Male,35,9,134498.54,119924.80,0
8377,5143,15778526,Bradshaw,719,Spain,Female,48,5,0.00,78563.66,0
2505,3156,15794493,Chimaijem,641,Spain,Male,32,7,0.00,24267.28,0


## Handling Against Imbalance and/or Skewed Data

* Imbalanced data are treated using stratify mechanism and `class_weight` parameter available for sklearn's `HistGradientBoostingClassifier`
* No additional treatment on skewed data since we are going to use Tree-based ensembles.

## Saving preprocessed data

In [10]:
if not os.path.exists('data/preprocessed'):
    os.makedirs('data/preprocessed')

with open('data/preprocessed/data.pkl', 'wb') as file:
    dump(in_df, file)